# Fourier expressivity across QASP models

Analyze both classifier readouts, RL policies, a Q-function, and fidelity/projected kernels. All general experiments initialize circuit angles and input weights independently through QASP's `Uniform(-np.pi, np.pi)` and `Normal(0, 1)` initializers in config dictionaries. The scalar analytic benchmark is the only fixed-initialization example, so its coefficients can be checked against a formula.

Run cells in order with a Python kernel containing this source checkout. Install from the repository root with `python -m pip install -e ".[dev]"`. The analyzer is a development addition to v0.1.3. Simulations use CPU statevectors and analytic measurements. Edit `neural_config`, `q_function`, or `kernel_config` to change the experiments. Each model is sampled once and reused for its direct and Fourier evaluations. The seed makes the notebook reproducible; the sweep contains one random model per width, not an average over random initializations.

In [ ]:
%matplotlib inline
from math import comb
from pathlib import Path
import sys

# Allow Run All directly from the tutorial directory in this checkout.
for parent in (Path.cwd(), *Path.cwd().parents):
    if (parent / "qasp/diagnostics/expressivity.py").is_file():
        sys.path.insert(0, str(parent))
        break

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from matplotlib.colors import Normalize
from IPython.display import display
from qasp import (
    AngleAnsatz, NeuralEmbedding, Normal, Uniform, Ones, Zeros, create_classifier, create_kernel,
    create_model, create_qfunction, expressivity_analyzer,
)
from qasp.rl import LearnablePQCBorn, PQCBorn

WIDTHS = [2, 3, 4, 5]                 # Joint fidelity costs 3**(2*n) evaluations here.
N_FEATURES = 2
REFERENCE = np.array([0.33, -0.51])
X = np.array([[-1.21, 0.32], [-0.43, -0.76], [0.17, 0.91], [0.64, -0.29], [1.38, 0.57]])
U = X[:, :1]                         # Scalar inputs for the analytic example and Q-function.
CHECK_ATOL = 1e-9
KERNEL_ATOL = 2e-6  # QASP trainable kernel APIs convert inputs and circuit arithmetic to float32.
assert WIDTHS == sorted(set(WIDTHS)) and min(WIDTHS) >= 2
torch.set_default_dtype(torch.float64)  # Initialize directly at the precision used by the Fourier checks.
torch.manual_seed(17)
np.random.seed(17)
errors = {}

## Reading the plots

The analyzer uses $f_a(x)=\sum_\omega c_{\omega,a}\exp(-i\omega\cdot x)$. **Allowed** includes every signed generator frequency, including zero. **Active** counts coefficients whose magnitude exceeds $10^{-10}$ for a particular readout.

Independent random input weights give each wire its own frequency direction. The general plots show frequency counts across widths and the coefficient spectrum at the largest width. Each point represents one frequency vector; its horizontal coordinate is $\|\omega\|_2$, so different vectors can share that coordinate. Final plots compare coefficient statistics across widths, and track the DC coefficient, whose physical frequency stays zero. Random nonzero frequencies generally change with width.

With one independently scaled encoding per wire, the phase grid has $3^n$ points for a single input and $3^{2n}$ for joint fidelity. The default joint sweep stops at five qubits to stay under the analyzer's 65,536-evaluation limit. The fixed scalar benchmark has a much smaller grid. Change `WIDTHS` with these different costs in mind.

In [ ]:
def neural_config(n):
    # QASP independently samples every circuit angle and neural input weight.
    return {
        "feature_map": NeuralEmbedding,
        "feature_map_config": {"n_qubits": n, "features": N_FEATURES, "rotations": "Y"},
        "ansatz": AngleAnsatz,
        "ansatz_config": {"n_qubits": n, "n_layers": 1, "rotations": "Y"},
        "init": Uniform(-np.pi, np.pi),
        "input_scaling_init": Normal(0, 1),
        "device": "default.qubit",
    }


def z_readouts(n):
    return ["Z 0", "Z" * n + " " + " ".join(map(str, range(n)))]


def forward_rows(model, points):
    with torch.no_grad():
        return np.stack([model(torch.tensor(p, dtype=torch.float64)).numpy().reshape(-1)
                         for p in points])


def check(name, width, reconstructed, direct, atol=CHECK_ATOL):
    reconstructed, direct = np.asarray(reconstructed), np.asarray(direct)
    assert reconstructed.shape == direct.shape, (name, reconstructed.shape, direct.shape)
    np.testing.assert_allclose(reconstructed, direct, rtol=0, atol=atol, err_msg=name)
    errors[name, width] = float(np.max(np.abs(reconstructed - direct)))


models_by_experiment = {}


def checked_sweep(name, factory, points=X, direct=forward_rows, coordinates=None, check_atol=CHECK_ATOL, **options):
    models = {n: factory(n) for n in WIDTHS}
    models_by_experiment[name] = models
    sweep = expressivity_analyzer(model_factory=models.__getitem__, qubit_list=WIDTHS, **options)
    for n, spectrum in sweep.results_by_qubit.items():
        encoded = points if coordinates is None else coordinates(points)
        check(name, n, spectrum.evaluate(encoded), direct(models[n], points), atol=check_atol)
    return sweep

In [ ]:
def harmonic_numbers(frequencies, direction):
    direction = np.asarray(direction, dtype=float)
    modes = np.rint((frequencies @ direction) / (direction @ direction)).astype(int)
    np.testing.assert_allclose(frequencies, modes[:, None] * direction, atol=1e-13, rtol=0)
    return modes


def plot_analytic_sweep(sweep, direction, title, output_names):
    spectra = list(sweep.results_by_qubit.values())
    widths = sweep.qubit_list
    modes_by_width = [harmonic_numbers(s.frequencies, direction) for s in spectra]
    limit = max(abs(m).max() for m in modes_by_width)
    modes = np.arange(-limit, limit + 1)
    norm = Normalize(0, max(s.magnitudes.max() for s in spectra) or 1)
    cmap = plt.get_cmap("magma").copy()
    cmap.set_bad("#dadde3")
    colors = ["#007b78", "#b14f27"]

    fig, axes = plt.subplots(1, 1 + len(output_names), figsize=(3.9 * (1 + len(output_names)), 3),
                             constrained_layout=True)
    fig.suptitle(title, fontweight="bold")
    count_ax = axes[0]
    count_ax.plot(widths, sweep.frequency_counts, "o--", color="#34383d", label="Allowed")
    active = np.stack([s.active_counts for s in spectra])
    for a, label in enumerate(output_names):
        count_ax.plot(widths, active[:, a], marker=["s", "^"][a], color=colors[a],
                      linestyle=["-", ":"][a], label=label, markersize=5)
        grid = np.full((len(widths), len(modes)), np.nan)
        for i, (s, harmonics) in enumerate(zip(spectra, modes_by_width)):
            grid[i, harmonics + limit] = s.magnitudes[:, a]
        ax = axes[a + 1]
        image = ax.imshow(grid, origin="lower", interpolation="nearest", aspect="auto",
                          cmap=cmap, norm=norm, extent=[-limit - .5, limit + .5, -.5, len(widths) - .5])
        ax.set(title=label, xlabel="Signed harmonic m", ylabel="Data qubits",
               yticks=np.arange(len(widths)), yticklabels=widths)
        ax.set_xticks(np.unique(np.rint(np.linspace(-limit, limit, min(len(modes), 7))).astype(int)))
    count_ax.set(title="Frequency counts", xlabel="Data qubits", ylabel="Signed frequencies", xticks=widths)
    count_ax.yaxis.set_major_locator(MaxNLocator(integer=True, nbins=6))
    count_ax.legend(frameon=False, fontsize=8)
    fig.colorbar(image, ax=list(axes[1:]), label=r"$|c_\omega|$", shrink=.83)
    plt.show()


def plot_sweep(sweep, title, output_names):
    spectra = list(sweep.results_by_qubit.values())
    widths = sweep.qubit_list
    largest = spectra[-1]
    fig, axes = plt.subplots(1, 1 + len(output_names),
                             figsize=(4 * (1 + len(output_names)), 3.2), constrained_layout=True)
    fig.suptitle(title, fontweight="bold")
    axes[0].plot(widths, sweep.frequency_counts, "o--", color="#34383d", label="Allowed")
    for a, label in enumerate(output_names):
        axes[0].plot(widths, [s.active_counts[a] for s in spectra], "o-", label=label)
        ax = axes[a + 1]
        ax.scatter(np.linalg.norm(largest.frequencies, axis=1), largest.magnitudes[:, a],
                   s=12, alpha=.6, color=["#007b78", "#b14f27"][a])
        ax.set(title=f"{label} · {widths[-1]} qubits", xlabel=r"Frequency norm $\|\omega\|_2$",
               ylabel=r"Coefficient magnitude $|c_\omega|$")
        ax.grid(alpha=.2)
    axes[0].set(title="Frequency counts", xlabel="Data qubits", ylabel="Signed frequencies", xticks=widths)
    axes[0].set_yscale("log")
    axes[0].legend(frameon=False, fontsize=8)
    plt.show()

## 1. Classifiers

### Local $Z$ and full-register parity

Start with the scalar circuit $R_Y(x)^{\otimes n}$. Local $Z$ gives $\cos x$. Full-register parity gives $\cos^n x$, with $c_{n-2j}=2^{-n}\binom{n}{j}$. We check **every complex coefficient** and reconstruct both expectations at five off-grid inputs for each width.

In [ ]:
def scalar_classifier(n):
    config = {
        "feature_map": "NeuralEmbedding", "feature_map_config": {"features": 1, "rotations": "Y"},
        "ansatz": "AngleAnsatz", "ansatz_config": {"n_layers": 1, "rotations": "Y"},
        "init": Zeros(), "input_scaling_init": Ones(),
        "readout": "expectation", "device": "default.qubit",
    }
    return create_classifier(config, n_qubits=n, n_classes=2)


def analytic_readouts(model, points):
    local = np.cos(points[:, 0])
    return np.column_stack([local, local ** model.n_qubits])


anchor = checked_sweep("Classifier • analytic observables", scalar_classifier, U,
                       direct=analytic_readouts, observables=z_readouts)
for n, s in anchor.results_by_qubit.items():
    m = harmonic_numbers(s.frequencies, [1.])
    np.testing.assert_array_equal(m, np.arange(-n, n + 1))
    local = np.where(abs(m) == 1, 0.5, 0.0)
    parity = [comb(n, (n - h) // 2) / 2**n if (n - h) % 2 == 0 else 0. for h in m]
    np.testing.assert_allclose(s.coefficients, np.column_stack([local, parity]), atol=1e-12)
    np.testing.assert_array_equal(s.active_counts, [2, n + 1])
plot_analytic_sweep(anchor, [1.], "Classifier · scalar benchmark", ["Local Z", "Parity"])

### Random class scores and class probabilities

Each wire gets independent random neural input weights and a random variational rotation. The expectation classifier averages local and parity observables and scales its score by 1.25. The probability classifier assigns even and odd parity outcomes to two classes. Each check calls the same initialized classifier's forward method.

In [ ]:
def make_classifier(n, readout):
    config = {**neural_config(n), "readout": readout}
    if readout == "expectation":
        config.update(observables=z_readouts(n), output_scaling=1.25)
    else:
        config.update(decoder="parity", measurement_qubits=list(range(n)))
    return create_classifier(config, n_qubits=n, n_classes=2)


scores = checked_sweep("Classifier • score", lambda n: make_classifier(n, "expectation"))
classes = checked_sweep("Classifier • probabilities", lambda n: make_classifier(n, "probabilities"))
plot_sweep(scores, "Classifier · expectation score", ["Class score"])
plot_sweep(classes, "Classifier · parity probabilities", ["Class 0", "Class 1"])

## 2. RL policies

The softmax policy exposes action logits. Its affine head uses the constructor's random initialization, so its weights and bias also enter the coefficients. The Born policy returns decoded even/odd action probabilities. Reconstruct each output vector at identical inputs and check that the Born probabilities sum to one.

In [ ]:
def softmax_policy(n):
    config = {
        **neural_config(n), "n_actions": 2, "observables": z_readouts(n),
        "fc_layer": True, "output_scaling": 1.2,
    }
    return create_model("PQCSoftmax", config, n_qubits=n)


def born_policy(n):
    config = {
        **neural_config(n), "n_actions": 2, "policy": "parity-like",
        "training_method": "probs", "parity_cnot_decomposition": False,
    }
    return create_model("PQCBorn", config, n_qubits=n)


logits = checked_sweep("Policy • logits", softmax_policy)
actions = checked_sweep("Policy • Born probabilities", born_policy)
for s in actions.results_by_qubit.values():
    np.testing.assert_allclose(s.evaluate(X).sum(axis=1), 1., atol=1e-12)
plot_sweep(logits, "Softmax policy · forward logits", ["Action 0 logit", "Action 1 logit"])
plot_sweep(actions, "Born policy · decoded probabilities", ["Action 0", "Action 1"])

### Legacy and adaptive Born policies

The `create_model("PQCBorn", ...)` factory above selects the current decoder policy. Use `PQCBorn(**config)` and `LearnablePQCBorn(**config)` to select the legacy and adaptive classes explicitly.

For the legacy sampler, the finite readout is the pair of basis probabilities on wire 0. For the adaptive policy, fix the observable before analysis. Here we request local Z and parity within that subset and independently recover both from the policy's circuit probabilities on the first selectable half-register. The learned, stochastic wire selector is outside this fixed-observable check.

In [ ]:
def legacy_policy(n):
    config = {**neural_config(n), "n_qubits": n, "n_actions": 2, "measurement_qubits": [0]}
    return PQCBorn(**config)


def adaptive_policy(n):
    config = {**neural_config(n), "n_qubits": n, "n_actions": 2, "n_qubits_measure": n // 2}
    return LearnablePQCBorn(**config)


def adaptive_moments(model, points):
    selector = model.measurement_combinations[0]
    n = len(selector)
    basis = np.arange(2**n)
    local = 1 - 2 * (basis >> (n - 1))
    parity = [(-1)**int(i).bit_count() for i in basis]
    selector = np.asarray(selector)
    with torch.no_grad():
        probabilities = np.stack([model.qlayer(torch.tensor(np.r_[p, selector])).numpy()
                                  for p in points])
    return probabilities @ np.column_stack([local, parity])


legacy = checked_sweep("Policy • legacy basis", legacy_policy,
                       direct=lambda model, points: forward_rows(model.qlayer, points))
adaptive = checked_sweep("Policy • adaptive fixed observables", adaptive_policy,
                         direct=adaptive_moments, observables=lambda n: z_readouts(n//2))
plot_sweep(legacy, "Legacy Born policy · measured wire 0", ["Basis 0", "Basis 1"])
plot_sweep(adaptive, "Adaptive Born policy · fixed subset", ["Local Z", "Subset parity"])

## 3. Q-functions

Use a scaled `AngleEmbedding` with two uploads of one scalar on wire 0. QASP samples both upload scales independently with `Normal(0, 1)` and every variational angle with `Uniform(-np.pi, np.pi)`. The Fourier variable is the encoded state $x=\tanh(s)$. Two generic random scales produce nine allowed signed frequencies. Their locations and coefficients change between models.

In [ ]:
def q_function(n):
    config = {
        "feature_map": "AngleEmbedding",
        "feature_map_config": {"features": 1, "rotations": ["Y 0"], "input_scaling": True},
        "ansatz": "AngleAnsatz",
        "ansatz_config": {"n_layers": 2, "rotations": "YX", "reuploading": True},
        "observables": z_readouts(n), "fc_layer": True, "fc_bias": True,
        "input_activation": "tanh", "init": Uniform(-np.pi, np.pi),
        "input_scaling_init": Normal(0, 1),
        "device": "default.qubit",
    }
    return create_qfunction(config, n_qubits=n, n_actions=2)


qvals = checked_sweep("Q-function • affine Q-values", q_function, U, coordinates=np.tanh)
np.testing.assert_array_equal(qvals.frequency_counts, np.full(len(WIDTHS), 9))
plot_sweep(qvals, "Q-function · scaled angle reuploads", ["Q(state, 0)", "Q(state, 1)"])

## 4. Fidelity kernels

Construct a trainable feature circuit with random variational angles and random neural input weights. First fix $y=(0.33,-0.51)$ and analyze $k(x,y)$ as a function of $x$. Then analyze both inputs jointly, reusing those same initialized kernels.

For this one-upload architecture the variational block follows the encoding. It cancels between the two states in global fidelity, even though its parameters are randomly initialized. The projected kernel below remains sensitive to that block. Joint fidelity still explores two independent sets of input phases, which makes its extraction more expensive.

In [ ]:
def kernel_config(n):
    return {
        "feature_map": "NeuralEmbedding",
        "feature_map_config": {"features": N_FEATURES, "rotations": "Y"},
        "ansatz": "AngleAnsatz",
        "ansatz_config": {"n_layers": 1, "rotations": "YX"},
        "init": Uniform(-np.pi, np.pi),
        "input_scaling_init": Normal(0, 1),
        "device": "default.qubit",
    }


def fidelity(n):
    config = {**kernel_config(n), "kernel_type": "fidelity"}
    return create_kernel(config, n_qubits=n)


conditional = checked_sweep("Fidelity • fixed reference", fidelity, reference=REFERENCE, check_atol=KERNEL_ATOL,
                            direct=lambda k, points: np.array([[k.kernel(x, REFERENCE)] for x in points]))
plot_sweep(conditional, "Fidelity kernel · fixed reference", ["Kernel value"])

Y = X[::-1] + np.array([0.17, -0.09])
fidelity_models = models_by_experiment["Fidelity • fixed reference"]
joint = expressivity_analyzer(model_factory=fidelity_models.__getitem__, qubit_list=WIDTHS)
for n, s in joint.results_by_qubit.items():
    k = fidelity_models[n]
    expected = np.array([[k.kernel(x, y)] for x, y in zip(X, Y)])
    check("Fidelity • joint", n, s.evaluate(X, Y), expected, atol=KERNEL_ATOL)
    assert s.n_frequencies == 3**(2*n)

The joint spectrum has four coordinates $(\omega_{x,0},\omega_{x,1},\omega_{y,0},\omega_{y,1})$. The scatter below uses their Euclidean norm and includes all coefficients. The count plot compares allowed and active joint frequencies across the complete sweep.

In [ ]:
plot_sweep(joint, "Fidelity kernel · joint inputs", ["Kernel value"])

## 5. Projected kernels

The finite spectra here belong to the measured local and parity features. Reconstruct both features first, apply the kernel's weighted Gaussian to their differences, and compare the entire $5\times5$ cross-Gram matrix to `gram(X, Y)` at every width.

In [ ]:
def projected(n):
    config = {
        **kernel_config(n), "kernel_type": "projected",
        "observables": z_readouts(n), "gamma": 0.8, "weights": [0.35, 1.1],
    }
    return create_kernel(config, n_qubits=n)


projection = checked_sweep("Projected kernel • features", projected, direct=lambda k, points: k.features(points), check_atol=KERNEL_ATOL)
for n, s in projection.results_by_qubit.items():
    k = models_by_experiment["Projected kernel • features"][n]
    delta = s.evaluate(X)[:, None, :] - s.evaluate(Y)[None, :, :]
    fourier_gram = np.exp(-k.gamma * np.sum(k.weights * delta**2, axis=-1))
    check("Projected kernel • cross-Gram", n, fourier_gram, k.gram(X, Y), atol=KERNEL_ATOL)
plot_sweep(projection, "Projected kernel · measured features", ["Local Z", "Parity"])

### Random neural projections with entanglement

Each wire again has independent random input weights. Add a nearest-neighbor CNOT chain to the feature map and analyze the final-wire Z readout. Its frequencies are combinations of the sampled directions, without imposing a shared projection or integer harmonic lattice.

In [ ]:
def entangled_kernel(n):
    config = {
        **kernel_config(n),
        "kernel_type": "projected",
        "feature_map_config": {
            "features": N_FEATURES, "rotations": "Y",
            "imprimitive": "CNOT", "imprimitive_pattern": "chain",
        },
        "observables": [f"Z {n-1}"],
    }
    return create_kernel(config, n_qubits=n)


plane = checked_sweep("Projected kernel • entangled", entangled_kernel,
                      direct=lambda k, points: k.features(points), check_atol=KERNEL_ATOL)
plot_sweep(plane, "Projected kernel · random entangled encoding", ["Final-wire Z"])

## Coefficients as a function of qubit count

Randomly sampled input weights change the nonzero frequency locations at each width. The following panels therefore show the **largest non-DC magnitude** and the **root-sum-square of all non-DC magnitudes**, together with the real and imaginary parts of the **DC coefficient**. DC always refers to the same physical frequency, zero. The largest non-DC coefficient may belong to a different frequency at each width. Each panel names the output being analyzed; edit the output index in `tracks` to inspect another one.

These curves describe the sampled models. Both circuit size and initialization differ across widths, so a trend should not be interpreted as a seed-averaged effect of qubit count alone.

In [ ]:
# Title, sweep, and output column.
tracks = [
    ("Scalar classifier · parity", anchor, 1),
    ("Classifier · score", scores, 0),
    ("Classifier · class 0", classes, 0),
    ("Softmax policy · logit 1", logits, 1),
    ("Born policy · action 0", actions, 0),
    ("Legacy policy · basis 0", legacy, 0),
    ("Adaptive policy · subset parity", adaptive, 1),
    ("Q-function · action 1", qvals, 1),
    ("Fidelity · fixed reference", conditional, 0),
    ("Fidelity · joint", joint, 0),
    ("Projected kernel · parity", projection, 1),
    ("Entangled kernel · final-wire Z", plane, 0),
]

fig, axes = plt.subplots(4, 3, figsize=(12, 10), sharex=True, constrained_layout=True)
for ax, (title, sweep, output) in zip(axes.flat, tracks):
    dc, peak, norm = [], [], []
    for s in sweep.results_by_qubit.values():
        zero = np.all(s.frequencies == 0, axis=1)
        assert zero.sum() == 1
        dc.append(s.coefficients[zero, output].item())
        non_dc = s.magnitudes[~zero, output]
        peak.append(non_dc.max(initial=0))
        norm.append(np.linalg.norm(non_dc))
    dc = np.asarray(dc)
    ax.axhline(0, color="#c9cdd2", lw=.8)
    ax.plot(sweep.qubit_list, peak, "o-", color="#252525", label="Max non-DC magnitude")
    ax.plot(sweep.qubit_list, norm, "d-", color="#7564a0", label="Non-DC root-sum-square")
    ax.plot(sweep.qubit_list, dc.real, "s--", color="#007b78", label="DC real part")
    ax.plot(sweep.qubit_list, dc.imag, "^:", color="#b14f27", label="DC imaginary part")
    ax.set(title=title, xticks=sweep.qubit_list)
    ax.title.set_fontsize(10)
    ax.grid(axis="y", alpha=.2)
for ax in axes[-1]:
    ax.set_xlabel("Data qubits")
for ax in axes[:, 0]:
    ax.set_ylabel("Coefficient statistic")
fig.legend(*axes[0, 0].get_legend_handles_labels(), loc="outside lower center", ncol=4, fontsize=9)
plt.show()

## Check the reconstructions

The table reports the maximum absolute error across all widths, test inputs, and output components in each check. Thresholds are set in the first code cell. Trainable kernel APIs use float32 inputs internally, so their comparisons allow `KERNEL_ATOL=2e-6`; other checks use `CHECK_ATOL=1e-9`. The Fourier extraction itself uses float64. To inspect a full complex spectrum, the last table lists the randomly initialized projected kernel’s local and parity coefficients at the largest width.

In [ ]:
validation = pd.Series(errors, name="max_abs_error").rename_axis(["Check", "Qubits"]).unstack()
validation["Worst error"] = validation.max(axis=1)
validation["Tolerance"] = [KERNEL_ATOL if "kernel" in name.lower() or "Fidelity" in name
                           else CHECK_ATOL for name in validation.index]
display(validation.style.format("{:.2e}"))
assert (validation["Worst error"] <= validation["Tolerance"]).all()

s = projection.results_by_qubit[max(WIDTHS)]
display(pd.DataFrame({"Frequency": list(map(tuple, s.frequencies)), "Local Z coefficient": s.coefficients[:, 0],
                      "Parity coefficient": s.coefficients[:, 1], "Parity magnitude": s.magnitudes[:, 1]})
        .set_index("Frequency").round(8))